# Install Dependecies

In [1]:
%%capture
%pip install -q "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [3]:
import numpy as np
import pandas as pd
import random
import torch
from collections import Counter
from datasets import load_dataset
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.std import tqdm
from transformers import AutoTokenizer
from transformers import AutoModel
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

# Import datasets

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [5]:
# the group language for week 38
LANGUAGE = "ko"

In [6]:
# select language
df_train = df_train[df_train["lang"] == LANGUAGE].copy()
df_val = df_val[df_val["lang"] == LANGUAGE].copy()

# Tokeniser

In [7]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 4 Week 38: Language Modelling
As a group, choose one language from Arabic, Korean or Telugu for this stage. Each group member must implement and analyse one language model or model configuration on this same language. You may use n-gram or neural language models; members may use the same model family while varying a modelling choice, such as n-gram order or smoothing. Include a simple baseline for comparison, also if you are working alone.

Use the same training questions, validation questions, tokenisation, vocabulary and unknown-token strategy for all models being compared. Train only on questions from the training split, build the shared vocabulary from that split, and report per-token perplexity on the corresponding validation questions. Describe your tokenisation and unknown-token strategy explicitly.

Compare the models' validation perplexities and discuss which modelling choices help. Inspect examples with high and low perplexity and formulate one hypothesis about what each model captures. Raw perplexities across different languages, tokenisations or vocabularies are not directly comparable; the shared setup above is required for this comparison.

In [8]:
train_questions = df_train["question"].drop_duplicates().tolist()
val_questions = df_val["question"].drop_duplicates().tolist()
print("train:", len(train_questions), "| val:", len(val_questions))

train: 2397 | val: 292


In [9]:
MIN_COUNT = 2
train_tokens = [tokenizer.tokenize(question) for question in train_questions]
val_tokens = [tokenizer.tokenize(question) for question in val_questions]

piece_counts = Counter(piece for tokens in train_tokens for piece in tokens)
vocabulary = {piece for piece, count in piece_counts.items() if count >= MIN_COUNT}

In [10]:
def replace_unknown(tokens):
    return [token if token in vocabulary else "<unk>" for token in tokens]

In [11]:
train_sentences = [replace_unknown(tokens) for tokens in train_tokens]
val_sentences = [replace_unknown(tokens) for tokens in val_tokens]

In [12]:
print("vocabulary:", len(vocabulary), "pieces + <unk> + </s>")
for name, sentences in [("train", train_sentences), ("val", val_sentences)]:
    tokens = [token for sentence in sentences for token in sentence]
    print(f"{name}: tokens: {len(tokens)} | <unk> share: {tokens.count('<unk>') / len(tokens):.3f}")

vocabulary: 1519 pieces + <unk> + </s>
train: tokens: 29090 | <unk> share: 0.024
val: tokens: 3520 | <unk> share: 0.050


### Models
(claude we used to produce this markdown cell)
* **Unigram (baseline)**: $P(w) = c(w) / N$
* **Bigram**: $P(w \mid v) = \lambda \, c(v, w) / c(v) + (1 - \lambda) \, P(w)$. Interpolating with the unigram keeps unseen bigrams above zero. $\lambda = 0$ gives the unigram baseline.

In [13]:
# bigram
unigram_counts = Counter()
bigram_counts = Counter()
context_counts = Counter()
for sentence in train_sentences:
    padded = ["<s>"] + sentence + ["</s>"]
    unigram_counts.update(padded[1:])
    bigram_counts.update(zip(padded, padded[1:]))
    context_counts.update(padded[:-1])
unigram_total = sum(unigram_counts.values())

In [14]:
def unigram_prob(token):
    return unigram_counts[token] / unigram_total

In [15]:
def bigram_prob(previous, token, lam):
    return lam * bigram_counts[(previous, token)] / context_counts[previous] + (1 - lam) * unigram_prob(token)

In [16]:
# def trigram_prob(before, previous, token, lam):
#     if context_counts[(before, previous)] == 0:
#         return unigram_prob(token)
#     return lam * trigram_counts[(before, previous, token)] / context_counts[(before, previous)] + (1 - lam) * unigram_prob(token)

In [17]:
# bigram
def perplexity(sentences, lam):
    log_prob = 0.0
    n_tokens = 0
    for sentence in sentences:
        padded = ["<s>"] + sentence + ["</s>"]
        for previous, token in zip(padded, padded[1:]):
            log_prob += np.log(bigram_prob(previous, token, lam))
            n_tokens += 1
    return np.exp(-log_prob / n_tokens)

### Choosing $\lambda$
$\lambda$ is chosen on the validation questions (shared protocol: validation split for model selection). $\lambda = 1$ is left out because unseen bigrams would get probability 0.

In [18]:
lambda_scores = pd.DataFrame({"lambda": [round(lam, 1) for lam in np.arange(0.0, 1.0, 0.1)]})
lambda_scores["val_perplexity"] = [perplexity(val_sentences, lam) for lam in lambda_scores["lambda"]]
best_lambda = lambda_scores.loc[lambda_scores["val_perplexity"].idxmin(), "lambda"]
print("best lambda:", best_lambda)
lambda_scores.round(1)

best lambda: 0.8


,lambda,val_perplexity
0,0.0,231.9
1,0.1,79.0
2,0.2,56.0
3,0.3,45.1
4,0.4,38.7
5,0.5,34.7
6,0.6,32.0
7,0.7,30.5
8,0.8,30.1
9,0.9,31.7


### Results

In [19]:
# bigram
results = pd.DataFrame({
    "model": ["unigram", "bigram"],
    "lambda": [0.0, best_lambda],
    "train_perplexity": [perplexity(train_sentences, 0.0), perplexity(train_sentences, best_lambda)],
    "val_perplexity": [perplexity(val_sentences, 0.0), perplexity(val_sentences, best_lambda)],
})
results.round(1)

,model,lambda,train_perplexity,val_perplexity
0,unigram,0.0,245.6,231.9
1,bigram,0.8,9.7,30.1


### High and low perplexity questions (val)

In [20]:
# bigram
df_ppl = pd.DataFrame({"question": val_questions, "tokens": val_sentences})
df_ppl["n_unk"] = [sentence.count("<unk>") for sentence in val_sentences]
df_ppl["unigram"] = [perplexity([sentence], 0.0) for sentence in val_sentences]
df_ppl["bigram"] = [perplexity([sentence], best_lambda) for sentence in val_sentences]

In [21]:
# unigram: lowest and highest
display(df_ppl.sort_values("unigram").head(5).round(1))
display(df_ppl.sort_values("unigram").tail(5).round(1))

,question,tokens,n_unk,unigram,bigram
20,최초의 광고 대행 업체는 무엇인가?,"[▁최초, 의, <unk>, ▁, <unk>, <unk>, 는, ▁무엇인가, ?]",3,43.2,10.3
86,세상에서 가장 넓은 산은 무엇인가?,"[▁세상, 에서, ▁가장, <unk>, ▁산, 은, ▁무엇인가, ?]",1,63.0,8.9
218,세상에서 가장 큰 대학교는 무엇인가?,"[▁세상, 에서, ▁가장, ▁큰, ▁, 대학교, 는, ▁무엇인가, ?]",0,67.5,4.6
234,KKK의 지도자는 누구인가?,"[<unk>, <unk>, 의, ▁지도, 자는, ▁누구, 인가, ?]",2,69.3,5.3
206,창원시에서 가장 높은 건물은 무엇인가?,"[▁, <unk>, 시, 에서, ▁가장, ▁높은, ▁건물, 은, ▁무엇인가, ?]",1,78.0,10.1


,question,tokens,n_unk,unigram,bigram
170,부커 T. 워싱턴은 총 몇번의 결혼을 했나요?,"[▁부, 커, ▁T, ., ▁, 워, 싱, 턴, 은, ▁총, ▁몇, 번, 의, ▁결...",0,605.8,20.1
212,신화에는 그리스 신들이 몇 명이나 있을까?,"[▁신, 화, 에는, ▁그리, 스, ▁신, 들이, ▁몇, ▁명, 이나, ▁있을까, ?]",0,652.5,195.3
276,말레이시아에서는 핵무기가 합법적인가?,"[▁말, 레이, 시아, 에서는, ▁, 핵, 무, 기가, ▁합, 법, 적인, 가, ?]",0,793.1,16.2
145,인도네시아에서는 동성결혼이 합법적인가?,"[▁인도, 네, 시아, 에서는, ▁동, 성, 결, 혼, 이, ▁합, 법, 적인, 가...",0,879.5,52.5
52,더글러스 맥아더 배우자는 누군가요?,"[▁더, 글, 러, 스, ▁맥, 아, 더, ▁배우, 자는, ▁누군가, 요, ?]",0,928.2,627.2


In [22]:
# bigram
display(df_ppl.sort_values("bigram").head(5).round(1))
display(df_ppl.sort_values("bigram").tail(5).round(1))

,question,tokens,n_unk,unigram,bigram
263,중국에서 가장 인구밀도가 높은 도시는 어디인가?,"[▁중국, 에서, ▁가장, ▁인구, 밀, 도가, ▁높은, ▁도시, 는, ▁어디, 인...",0,153.7,4.0
173,스웨덴에서 가장 인구밀도가 높은 도시는 어디인가?,"[▁스, 웨, 덴, 에서, ▁가장, ▁인구, 밀, 도가, ▁높은, ▁도시, 는, ▁...",0,237.8,4.5
198,중국에서 가장 작은 도시는 어디인가?,"[▁중국, 에서, ▁가장, ▁작은, ▁도시, 는, ▁어디, 인가, ?]",0,96.8,4.5
218,세상에서 가장 큰 대학교는 무엇인가?,"[▁세상, 에서, ▁가장, ▁큰, ▁, 대학교, 는, ▁무엇인가, ?]",0,67.5,4.6
79,세상에서 가장 오래된 대학교는 무엇인가?,"[▁세상, 에서, ▁가장, ▁오래, 된, ▁, 대학교, 는, ▁무엇인가, ?]",0,92.2,4.6


,question,tokens,n_unk,unigram,bigram
104,제1대 휘트워스 준남작 조지프 휘트워스 경의 소속은 어디인가요?,"[▁제, 1, 대, ▁, 휘, 트, 워, 스, <unk>, 남, 작, ▁조, 지, ...",2,386.3,349.0
24,육파철학의 이름들은 무엇인가?,"[▁육, 파, 철, 학, 의, ▁이름, 들은, ▁무엇인가, ?]",0,536.8,387.3
181,불고기는 소의 어느부위를 사용하는가?,"[▁불, 고, 기는, ▁소, 의, ▁어느, 부, 위를, ▁사용하는, 가, ?]",0,539.2,456.4
195,코탕탱반도는 현재의 어디인가?,"[▁코, 탕, 탱, 반, 도는, ▁현재, 의, ▁어디, 인가, ?]",0,488.0,459.4
52,더글러스 맥아더 배우자는 누군가요?,"[▁더, 글, 러, 스, ▁맥, 아, 더, ▁배우, 자는, ▁누군가, 요, ?]",0,928.2,627.2
